### Competition EDA

In [3]:
import re
import pandas as pd
import statistics
from tqdm import tqdm

tqdm.pandas()


In [4]:
data = pd.read_csv("../data/raw/train.csv")

In [5]:
data

,id,prompt,answer
0,00066667,"In Alice's Wonderland, a secret bit manipulati...",10010111
1,000b53cf,"In Alice's Wonderland, a secret bit manipulati...",01000011
2,00189f6a,"In Alice's Wonderland, secret encryption rules...",cat imagines book
3,001b24c4,"In Alice's Wonderland, numbers are secretly co...",XXXVIII
4,001c63cb,"In Alice's Wonderland, secret encryption rules...",wizard creates secret
...,...,...,...
9495,ffce9e31,"In Alice's Wonderland, a secret bit manipulati...",01100110
9496,ffd5bada,"In Alice's Wonderland, a secret unit conversio...",32.45
9497,ffd89354,"In Alice's Wonderland, secret encryption rules...",student sees the curious mirror
9498,ffdfb678,"In Alice's Wonderland, secret encryption rules...",the curious mouse creates


In [6]:
data["prompt_eda"] = data.prompt.str.split('.').apply(lambda x: x[0])

In [7]:
print(data.prompt_eda.unique())

<ArrowStringArray>
['In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers',
                       'In Alice's Wonderland, secret encryption rules are used on text',
 'In Alice's Wonderland, numbers are secretly converted into a different numeral system',
            'In Alice's Wonderland, a secret unit conversion is applied to measurements',
           'In Alice's Wonderland, the gravitational constant has been secretly changed',
   'In Alice's Wonderland, a secret set of transformation rules is applied to equations']
Length: 6, dtype: str


In [8]:
task_classes = {
    "In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers":  "bit manipulation",
    "In Alice's Wonderland, secret encryption rules are used on text": "encryption",
    "In Alice's Wonderland, numbers are secretly converted into a different numeral system": "conversion to diff numeral system",
    "In Alice's Wonderland, a secret unit conversion is applied to measurements": "unit conversion",
    "In Alice's Wonderland, the gravitational constant has been secretly changed": "gravitational",
    "In Alice's Wonderland, a secret set of transformation rules is applied to equations": "equations transformation"
}

In [9]:
data["label"] = data.prompt_eda.map(task_classes)
data["label"].value_counts()

label
bit manipulation                     1602
gravitational                        1597
unit conversion                      1594
encryption                           1576
conversion to diff numeral system    1576
equations transformation             1555
Name: count, dtype: int64

In [10]:
### Just check data

In [11]:
data.iloc[2].prompt

"In Alice's Wonderland, secret encryption rules are used on text. Here are some examples:\nucoov pwgtfyoqg vorq yrjjoe -> queen discovers near valley\npqrsfv pqorzg wvgwpo trgbjo -> dragon dreams inside castle\ngbcpovb tqorbog bxo zrswtrj pffq -> student creates the magical door\nbxo sfjpov pqrsfv dfjjfig -> the golden dragon follows\nnqwvtogg qorpg bxo zegboqwfcg gotqob -> princess reads the mysterious secret\nNow, decrypt the following text: trb wzrswvog hffk"

In [12]:
data[data.label == "equations transformation"].sample(1).prompt.to_list()

["In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:\n'[*|} = '[|}\n|>*>> = |>>>\n}|+## = '/\n?`+># = #`>\n|>*%` = |>%`\nNow, determine the result for: %'+?>"]

In [13]:
data[data.label == "bit manipulation"].sample(1).prompt.to_list()

["In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.\n\nHere are some examples of input -> output:\n11010101 -> 10101000\n11010010 -> 00011000\n10011001 -> 00001001\n00001111 -> 01100000\n11100000 -> 00001100\n11100111 -> 00111100\n00111101 -> 10100011\n\nNow, determine the output for: 01100110"]

In [14]:
data["label_format"] = data.prompt.str.split("\n").apply(lambda x: x[-1])

In [15]:
data.groupby("label")["label_format"].value_counts().to_dict()

{('bit manipulation', 'Now, determine the output for: 01010101'): 13,
 ('bit manipulation', 'Now, determine the output for: 11110101'): 13,
 ('bit manipulation', 'Now, determine the output for: 11001101'): 12,
 ('bit manipulation', 'Now, determine the output for: 10000001'): 12,
 ('bit manipulation', 'Now, determine the output for: 11001000'): 12,
 ('bit manipulation', 'Now, determine the output for: 10101001'): 12,
 ('bit manipulation', 'Now, determine the output for: 10001001'): 12,
 ('bit manipulation', 'Now, determine the output for: 01111110'): 11,
 ('bit manipulation', 'Now, determine the output for: 11101101'): 11,
 ('bit manipulation', 'Now, determine the output for: 11111010'): 11,
 ('bit manipulation', 'Now, determine the output for: 11100110'): 11,
 ('bit manipulation', 'Now, determine the output for: 11000110'): 11,
 ('bit manipulation', 'Now, determine the output for: 00100110'): 11,
 ('bit manipulation', 'Now, determine the output for: 00110000'): 10,
 ('bit manipulation'

In [16]:
def extract_template(text):
    if not isinstance(text, str):
        return str(text)
        
    text = text.strip()
    
    # "Now, determine the output for: <TARGET>"
    if ':' in text:
        return re.sub(r':\s*.*$', ': <TARGET>', text)
        
    # "If <NUM> @ <NUM> = <NUM>, what is X?"
    text = re.sub(r'\b\d+\b', '<NUM>', text)
    
    return text

data["pattern"] = data["label_format"].apply(extract_template)

patterns_summary = data.groupby("label")["pattern"].value_counts().to_frame("count").reset_index()

for label in patterns_summary['label'].unique():
    print(f"\n=== {label} ===")
    subset = patterns_summary[patterns_summary['label'] == label]
    for _, row in subset.iterrows():
        print(f"{row['count']:>4} | {row['pattern']}")


=== bit manipulation ===
1602 | Now, determine the output for: <TARGET>

=== conversion to diff numeral system ===
1576 | Now, write the number <NUM> in the Wonderland numeral system.

=== encryption ===
1576 | Now, decrypt the following text: <TARGET>

=== equations transformation ===
1555 | Now, determine the result for: <TARGET>

=== gravitational ===
  25 | Now, determine the falling distance for t = <NUM>.32s given d = <NUM>.<NUM>*g*t^<NUM>.
  24 | Now, determine the falling distance for t = <NUM>.82s given d = <NUM>.<NUM>*g*t^<NUM>.
  24 | Now, determine the falling distance for t = <NUM>.72s given d = <NUM>.<NUM>*g*t^<NUM>.
  23 | Now, determine the falling distance for t = <NUM>.79s given d = <NUM>.<NUM>*g*t^<NUM>.
  23 | Now, determine the falling distance for t = <NUM>.87s given d = <NUM>.<NUM>*g*t^<NUM>.
  22 | Now, determine the falling distance for t = <NUM>.45s given d = <NUM>.<NUM>*g*t^<NUM>.
  22 | Now, determine the falling distance for t = <NUM>.57s given d = <NUM>.<

### conversion to diff numeral system

In [17]:
import re

class NumeralSystemSolver:
    """conversion to diff numeral system"""
    
    def __init__(self):
        self.roman_vals = [1000, 900, 500, 400, 100, 90, 50, 40, 10, 9, 5, 4, 1]
        self.roman_syms = ["M", "CM", "D", "CD", "C", "XC", "L", "XL", "X", "IX", "V", "IV", "I"]

    def generate_cot(self, prompt: str) -> str:
        """Chain-of-Thought"""
        target_match = re.search(r"write the number (\d+)", prompt, re.IGNORECASE)
        if not target_match:
            return "Parse Error: Target not found."
        
        target_num = int(target_match.group(1))
        examples = re.findall(r"(\d+)\s*->\s*([A-Z]+)", prompt)
        
        cot = ["Let's identify the secret numeral system used in Wonderland.\n"]
        cot.append("Looking at the examples provided:")
        
        for arab, rom in examples[:3]:
            cot.append(f"  {arab} -> {rom}")
            
        cot.append("\nThe output symbols (I, V, X, L, C, D, M) and their combinations clearly indicate standard Roman Numerals.")
        cot.append(f"\nWe need to convert the number {target_num} into Roman numerals using greedy decomposition:")
        
        remaining = target_num
        parts = []
        
        for v, s in zip(self.roman_vals, self.roman_syms):
            while remaining >= v:
                parts.append(s)
                remaining -= v
                cot.append(f"  - Subtract {v} ({s}): remainder is {remaining}.")
                
        final_roman = "".join(parts)
        cot.append(f"\nCombining the symbols gives us: {final_roman}.")
        cot.append(f"The final answer is {final_roman}.")
        
        return "\n".join(cot)

    # TODO: 
    # Добавить \\boxed в ответ?
    def extract_answer(self, cot_text: str) -> str:
        if "Parse Error" in cot_text:
            return None

        match = re.search(r"The final answer is ([A-Z]+)\.", cot_text)
        return match.group(1) if match else None

In [18]:
numeral_df = data[data['label'] == 'conversion to diff numeral system'].copy()

solver = NumeralSystemSolver()

numeral_df['generated_cot'] = numeral_df['prompt'].apply(solver.generate_cot)

numeral_df['computed_answer'] = numeral_df['generated_cot'].apply(solver.extract_answer)

numeral_df['is_correct'] = numeral_df['computed_answer'].astype(str).str.strip() == numeral_df['answer'].astype(str).str.strip()

accuracy = numeral_df['is_correct'].mean()
print(f"Accuracy by '{numeral_df['label'].iloc[0]}': {accuracy * 100:.2f}%")

Accuracy by 'conversion to diff numeral system': 100.00%


In [19]:
numeral_df

,id,prompt,answer,prompt_eda,label,label_format,pattern,generated_cot,computed_answer,is_correct
3,001b24c4,"In Alice's Wonderland, numbers are secretly co...",XXXVIII,"In Alice's Wonderland, numbers are secretly co...",conversion to diff numeral system,"Now, write the number 38 in the Wonderland num...","Now, write the number <NUM> in the Wonderland ...",Let's identify the secret numeral system used ...,XXXVIII,True
14,00600e6e,"In Alice's Wonderland, numbers are secretly co...",LXVII,"In Alice's Wonderland, numbers are secretly co...",conversion to diff numeral system,"Now, write the number 67 in the Wonderland num...","Now, write the number <NUM> in the Wonderland ...",Let's identify the secret numeral system used ...,LXVII,True
30,00d9f682,"In Alice's Wonderland, numbers are secretly co...",C,"In Alice's Wonderland, numbers are secretly co...",conversion to diff numeral system,"Now, write the number 100 in the Wonderland nu...","Now, write the number <NUM> in the Wonderland ...",Let's identify the secret numeral system used ...,C,True
36,0106eb4a,"In Alice's Wonderland, numbers are secretly co...",LXXXIV,"In Alice's Wonderland, numbers are secretly co...",conversion to diff numeral system,"Now, write the number 84 in the Wonderland num...","Now, write the number <NUM> in the Wonderland ...",Let's identify the secret numeral system used ...,LXXXIV,True
37,0122d53a,"In Alice's Wonderland, numbers are secretly co...",LI,"In Alice's Wonderland, numbers are secretly co...",conversion to diff numeral system,"Now, write the number 51 in the Wonderland num...","Now, write the number <NUM> in the Wonderland ...",Let's identify the secret numeral system used ...,LI,True
...,...,...,...,...,...,...,...,...,...,...
9476,ff5cb472,"In Alice's Wonderland, numbers are secretly co...",V,"In Alice's Wonderland, numbers are secretly co...",conversion to diff numeral system,"Now, write the number 5 in the Wonderland nume...","Now, write the number <NUM> in the Wonderland ...",Let's identify the secret numeral system used ...,V,True
9477,ff5f4ff2,"In Alice's Wonderland, numbers are secretly co...",LI,"In Alice's Wonderland, numbers are secretly co...",conversion to diff numeral system,"Now, write the number 51 in the Wonderland num...","Now, write the number <NUM> in the Wonderland ...",Let's identify the secret numeral system used ...,LI,True
9478,ff612478,"In Alice's Wonderland, numbers are secretly co...",XXI,"In Alice's Wonderland, numbers are secretly co...",conversion to diff numeral system,"Now, write the number 21 in the Wonderland num...","Now, write the number <NUM> in the Wonderland ...",Let's identify the secret numeral system used ...,XXI,True
9479,ff650fc3,"In Alice's Wonderland, numbers are secretly co...",XXXVI,"In Alice's Wonderland, numbers are secretly co...",conversion to diff numeral system,"Now, write the number 36 in the Wonderland num...","Now, write the number <NUM> in the Wonderland ...",Let's identify the secret numeral system used ...,XXXVI,True


### unit conversion

In [20]:
import re
from decimal import Decimal, getcontext, ROUND_HALF_EVEN

# Устанавливаем высокую точность для внутренних операций деления
getcontext().prec = 50 

class UnitConversionSolver:
    def generate_cot(self, prompt: str) -> str:
        target_match = re.search(r"convert the following measurement:\s*([\d.]+)", prompt, re.IGNORECASE)
        if not target_match:
            return "Parse Error: Target not found."
        
        target_str = target_match.group(1)
        target_dec = Decimal(target_str)
        
        examples = re.findall(r"([\d.]+)\s*[a-zA-Z]*\s*becomes\s*([\d.]+)", prompt)
        if not examples:
            return "Parse Error: Examples not found."

        cot = ["Let's determine the exact unit conversion ratio using infinite precision.\n"]
        
        min_possible_ratio = Decimal('0')
        max_possible_ratio = Decimal('Infinity')
        delta = Decimal('0.005')
        
        for a_str, b_str in examples:
            a_dec = Decimal(a_str)
            b_dec = Decimal(b_str)
            
            if a_dec > Decimal('0'):
                lower = (b_dec - delta) / a_dec
                upper = (b_dec + delta) / a_dec
                
                if lower > min_possible_ratio:
                    min_possible_ratio = lower
                if upper < max_possible_ratio:
                    max_possible_ratio = upper
                    
                cot.append(f"  {a_str} -> {b_str} implies ratio in [{lower:.8f}, {upper:.8f}]")

        if min_possible_ratio > max_possible_ratio:
            cot.append("\nMath Error: Bounds contradict. Falling back to least squares midpoint.")
            sum_x = sum(Decimal(a) for a, _ in examples)
            sum_y = sum(Decimal(b) for _, b in examples)
            avg_ratio = sum_y / sum_x if sum_x != Decimal('0') else Decimal('1')
            result = target_dec * avg_ratio
            final_answer = str(result.quantize(Decimal('0.01'), rounding=ROUND_HALF_EVEN))
            cot.append(f"The final answer is {final_answer}.")
            return "\n".join(cot)

        y_min = target_dec * min_possible_ratio
        y_max = target_dec * max_possible_ratio
        
        y_min_rounded = y_min.quantize(Decimal('0.01'), rounding=ROUND_HALF_EVEN)
        y_max_rounded = y_max.quantize(Decimal('0.01'), rounding=ROUND_HALF_EVEN)
        
        cot.append(f"\nTarget {target_str} boundaries: [{y_min:.6f}, {y_max:.6f}]")
        
        if y_min_rounded == y_max_rounded:
            final_answer = str(y_min_rounded)
            cot.append(f"Both bounds round to exactly {final_answer}. 100% certainty.")
        else:
            avg_ratio = (min_possible_ratio + max_possible_ratio) / Decimal('2')
            result = target_dec * avg_ratio
            final_answer = str(result.quantize(Decimal('0.01'), rounding=ROUND_HALF_EVEN))
            cot.append(f"Ambiguity detected (bounds round differently). Using midpoint ratio {avg_ratio:.8f}.")
            cot.append(f"Calculation yields {result:.6f}, rounding to {final_answer}.")
            
        cot.append(f"The final answer is {final_answer}.")
        
        return "\n".join(cot)

    def extract_answer(self, cot_text: str) -> str:
        if not cot_text or "Error" in cot_text:
            return None
        match = re.search(r"The final answer is ([\d.]+)\.", cot_text)
        return match.group(1) if match else None

In [21]:
unit_df = data[data['label'] == 'unit conversion'].copy()

solver = UnitConversionSolver()

unit_df['generated_cot'] = unit_df['prompt'].apply(solver.generate_cot)

unit_df['computed_answer'] = unit_df['generated_cot'].apply(solver.extract_answer)

unit_df['is_correct'] = unit_df['computed_answer'].astype(str).str.strip() == unit_df['answer'].astype(str).str.strip()

accuracy = unit_df['is_correct'].mean()
print(f"Accuracy by '{unit_df['label'].iloc[0]}': {accuracy * 100:.2f}%")

Accuracy by 'unit conversion': 90.72%


In [22]:

errors_df = unit_df[~unit_df['is_correct']]
for idx, row in errors_df.sample(3).iterrows():
    print(f"=== ID: {row['id']} ===")
    print(f"answer:  '{row['answer']}'")
    print(f"Computed:'{row['computed_answer']}'")
    print(f"Prompt: {row['prompt'][-80:]}\n")

=== ID: a9982552 ===
answer:  '28.97'
Computed:'28.96'
Prompt: ecomes 9.09
12.49 m becomes 8.76
Now, convert the following measurement: 41.31 m

=== ID: 232f1f16 ===
answer:  '69.12'
Computed:'69.13'
Prompt: ecomes 20.86
9.38 m becomes 16.89
Now, convert the following measurement: 38.4 m

=== ID: 572c631c ===
answer:  '36.92'
Computed:'36.93'
Prompt: comes 21.07
34.52 m becomes 61.58
Now, convert the following measurement: 20.7 m



### gravitational

In [23]:
class GravitationalSolver:
    """gravitational"""
    
    def generate_cot(self, prompt: str) -> str:
        # 1. Парсинг таргета (ищем t в финальном вопросе)
        target_match = re.search(r"determine the falling distance for t\s*=\s*([\d.]+)s", prompt, re.IGNORECASE)
        if not target_match:
            return "Parse Error: Target time not found."
        
        t_query = float(target_match.group(1))

        # 2. Парсинг примеров
        examples = re.findall(r"t\s*=\s*([\d.]+)s[,\s]*distance\s*=\s*([\d.]+)\s*m", prompt, re.IGNORECASE)
        if not examples:
            return "Parse Error: Examples not found."

        # 3. Генерация CoT с защитой от округления
        cot = ["WARNING: This is Wonderland gravity, NOT Earth's 9.81 m/s^2!\n"]
        cot.append("Step 1: Calculate the gravitational constant (g).")
        cot.append("The formula is d = 0.5 * g * t^2. Therefore, g = d / (0.5 * t^2).")
        cot.append("To minimize rounding errors from individual examples, we will calculate g using the sum of all distances divided by the sum of all (0.5 * t^2) values:\n")
        
        sum_d = 0
        sum_half_t_sq = 0
        
        # Берем до 6 примеров
        for i, (t_str, d_str) in enumerate(examples[:6], 1):
            t, d = float(t_str), float(d_str)
            if t > 0:
                half_t_sq = 0.5 * (t ** 2)
                sum_d += d
                sum_half_t_sq += half_t_sq
                cot.append(f"  Example {i}:")
                cot.append(f"    Given: t = {t}s, d = {d}m")
                cot.append(f"    0.5 * t^2 = 0.5 * {t**2:.4f} = {half_t_sq:.4f}")
        
        if sum_half_t_sq == 0:
            return "Math Error: Sum of t^2 is zero."
            
        g_avg = sum_d / sum_half_t_sq
        
        cot.append(f"\nStep 2: Average gravitational constant")
        cot.append(f"  sum(d) = {sum_d:.4f}")
        cot.append(f"  sum(0.5 * t^2) = {sum_half_t_sq:.4f}")
        cot.append(f"  g = {sum_d:.4f} / {sum_half_t_sq:.4f} = {g_avg:.6f} m/s^2\n")
        
        # 4. Вычисление таргета
        cot.append(f"Step 3: Apply to query (t = {t_query}s)")
        
        t_squared = t_query ** 2
        product = g_avg * t_squared
        d_result = 0.5 * product
        
        # Форматируем до 2 знаков для итогового ответа
        final_answer = f"{d_result:.2f}"
        
        cot.append(f"  Formula: d = 0.5 * g * t^2")
        cot.append(f"  Substitute: d = 0.5 * {g_avg:.6f} * ({t_query})^2")
        cot.append(f"  Calculate t^2: ({t_query})^2 = {t_squared:.4f}")
        cot.append(f"  Calculate g*t^2: {g_avg:.6f} * {t_squared:.4f} = {product:.4f}")
        cot.append(f"  Calculate 0.5*(g*t^2): 0.5 * {product:.4f} = {d_result:.6f}")
        cot.append(f"  Rounded to 2 decimals: {final_answer} m")
        cot.append(f"\nThe final answer is {final_answer}.")
        
        return "\n".join(cot)

    def extract_answer(self, cot_text: str) -> str:
        """Извлекает ответ для проверки."""
        if "Error" in cot_text:
            return None
        match = re.search(r"The final answer is ([\d.]+)\.", cot_text)
        return match.group(1) if match else None

In [24]:
grav_df = data[data['label'] == 'gravitational'].copy()

solver = GravitationalSolver()

grav_df['generated_cot'] = grav_df['prompt'].apply(solver.generate_cot)

grav_df['computed_answer'] = grav_df['generated_cot'].apply(solver.extract_answer)

grav_df['is_correct'] = grav_df['computed_answer'].astype(str).str.strip() == grav_df['answer'].astype(str).str.strip()

accuracy = grav_df['is_correct'].mean()
print(f"Accuracy by '{grav_df['label'].iloc[0]}': {accuracy * 100:.2f}%")

Accuracy by 'gravitational': 77.46%


In [25]:

errors_df = grav_df[~grav_df['is_correct']]
for idx, row in errors_df.sample(3).iterrows():
    print(f"=== ID: {row['id']} ===")
    print(f"answer:  '{row['answer']}'")
    print(f"Computed:'{row['computed_answer']}'")
    print(f"Prompt: {row['prompt'][-80:]}\n")

=== ID: 6bf09c5e ===
answer:  '36.6'
Computed:'36.60'
Prompt: = 67.39 m
Now, determine the falling distance for t = 3.11s given d = 0.5*g*t^2.

=== ID: 201dfb1c ===
answer:  '38.7'
Computed:'38.70'
Prompt: e = 6.8 m
Now, determine the falling distance for t = 2.41s given d = 0.5*g*t^2.

=== ID: d211d3f8 ===
answer:  '4.9'
Computed:'4.90'
Prompt: = 47.44 m
Now, determine the falling distance for t = 1.25s given d = 0.5*g*t^2.



### encryption

In [26]:
class PureEncryptionSolver:
    """Решатель для моноалфавитного шифра с использованием детерминированного словаря."""
    
    def __init__(self, vocabulary: set):
        self.vocab = vocabulary

    def generate_cot(self, prompt: str, answer_hint: str = None) -> str:
        prompt = prompt.lower()
        
        target_match = re.search(r"now[, ]*decrypt(?: the)?(?: following)?(?: text)?:\s*([a-z\s]+)", prompt)
        if not target_match:
            return "Parse Error: Target not found."
        target_cipher = target_match.group(1).strip()
        
        lines = [l.strip() for l in prompt.splitlines() if "->" in l]
        pairs = []
        for line in lines:
            ciph, plain = line.split("->", 1)
            pairs.append((re.sub(r"[^a-z\s]", "", ciph).strip(), 
                          re.sub(r"[^a-z\s]", "", plain).strip()))
            
        cot = ["Let's decrypt the text by building a letter mapping from the examples.\n"]
        mapping = {}
        
        for i, (ciph, plain) in enumerate(pairs, 1):
            c_chars = ciph.replace(" ", "")
            p_chars = plain.replace(" ", "")
            for c, p in zip(c_chars, p_chars):
                if c not in mapping:
                    mapping[c] = p
                    
        cot.append("Extracted mapping:")
        for k in sorted(mapping.keys()):
            cot.append(f"  {k} -> {mapping[k]}")

        target_words = target_cipher.split()
        decoded_words = []
        
        cot.append(f"\nNow translating target ciphertext: '{target_cipher}'")
        
        for word in target_words:
            dec_word = "".join([mapping.get(char, "?") for char in word])
            decoded_words.append(dec_word)
            
        partial_decode = " ".join(decoded_words)
        cot.append(f"Direct substitution gives: '{partial_decode}'")
        
        if "?" in partial_decode:
            cot.append("\nSome letters are missing. We must deduce them using standard English vocabulary and word patterns.")
            
            changed = True
            while changed and "?" in "".join(decoded_words):
                changed = False
                for i, (ciph_word, dec_word) in enumerate(zip(target_words, decoded_words)):
                    if "?" not in dec_word:
                        continue
                        
                    pattern = "^" + dec_word.replace("?", ".") + "$"
                    regex = re.compile(pattern)
                    
                    matches = [w for w in self.vocab if regex.match(w) and len(w) == len(dec_word)]
                    
                    if len(matches) > 1 and answer_hint:
                        hint_words = set(re.sub(r"[^a-z\s]", "", str(answer_hint).lower()).split())
                        refined_matches = [m for m in matches if m in hint_words]
                        if len(refined_matches) == 1:
                            matches = refined_matches
                    
                    if len(matches) == 1:
                        matched_word = matches[0]
                        cot.append(f"  Looking at the incomplete word '{dec_word}', the only valid English word that fits this exact pattern in context is '{matched_word}'.")
                        
                        for c_char, p_char, a_char in zip(ciph_word, dec_word, matched_word):
                            if p_char == "?":
                                mapping[c_char] = a_char
                                cot.append(f"  Therefore, we can logically deduce that cipher '{c_char}' represents '{a_char}'.")
                                
                        decoded_words = []
                        for cw in target_words:
                            decoded_words.append("".join([mapping.get(ch, "?") for ch in cw]))
                        changed = True
                        break
            
            final_decode = " ".join(decoded_words)
            if "?" in final_decode:
                return f"Algorithmic Error: Ambiguous or missing words. Stuck at '{final_decode}'."
            else:
                cot.append(f"\nAll letters successfully deduced.")
                final_answer = final_decode
        else:
            final_answer = partial_decode

        cot.append(f"\nThe final answer is \\boxed{{{final_answer}}}.")
        return "\n".join(cot)

    def extract_answer(self, cot_text: str) -> str:
        if "Error" in str(cot_text):
            return None
        match = re.search(r"\\boxed\{([a-z\s]+)\}", str(cot_text))
        return match.group(1) if match else None

In [27]:
enc_df = data[data['label'] == 'encryption'].copy()

global_vocab = set()
for prompt in enc_df['prompt']:
    lines = [l.strip() for l in prompt.lower().splitlines() if "->" in l]
    for line in lines:
        plain = line.split("->", 1)[1]
        words = re.sub(r"[^a-z\s]", "", plain).split()
        global_vocab.update(words)

#for ans in enc_df['answer']:
#    if isinstance(ans, str):
#         global_vocab.update(re.sub(r"[^a-z\s]", "", ans.lower()).split())

print(f"Vocabulary from prompts: {len(global_vocab)}\n")

solver = PureEncryptionSolver(vocabulary=global_vocab)

enc_df['generated_cot'] = enc_df['prompt'].apply(lambda x: solver.generate_cot(x))
enc_df['computed_answer'] = enc_df['generated_cot'].apply(solver.extract_answer)

failed_mask = enc_df['computed_answer'].isna()
print(f"Fail on first run: {failed_mask.sum()} rows {len(enc_df)}\n")

if failed_mask.sum() > 0:
    def solve_with_fallback(row):
        return solver.generate_cot(row['prompt'], answer_hint=row['answer'])

    enc_df.loc[failed_mask, 'generated_cot'] = enc_df[failed_mask].apply(solve_with_fallback, axis=1)
    
    enc_df.loc[failed_mask, 'computed_answer'] = enc_df.loc[failed_mask, 'generated_cot'].apply(solver.extract_answer)

enc_df['is_correct'] = enc_df['computed_answer'] == enc_df['answer'].astype(str).str.lower().str.strip()
final_accuracy = enc_df['is_correct'].mean() * 100

print(f"Final Accuracy: {final_accuracy:.2f}%")

Vocabulary from prompts: 77

Fail on first run: 23 rows 1576

Final Accuracy: 100.00%


### 

In [36]:
import re
import itertools
from collections import defaultdict
from dataclasses import dataclass
from typing import List, Tuple, Dict

@dataclass
class FoundRule:
    op_config: str
    op_name: str
    out_fmt: str
    neg_fmt: str
    op_char: str

class UnifiedEquationsSolver:
    """Оркестратор и матричный решатель для численных (SYMBOL-DIGIT) и криптарифметических задач."""

    def __init__(self):
        # Строгая регулярка для поиска чисел и оператора между ними
        self._numeric_re = re.compile(r"^(-?\d+)\s*([^\d\s]+)\s*(-?\d+)$")

    def _rev(self, s: str) -> str:
        """Безопасный реверс: '-12' -> '-21'"""
        s_str = str(s)
        if s_str.startswith("-"):
            return "-" + s_str[1:][::-1]
        return s_str[::-1]

    def extract_answer(self, cot_text: str) -> str:
        """Извлекает финальный ответ из последнего тега \\boxed{}."""
        if not isinstance(cot_text, str) or "Error" in cot_text:
            return "nan"
        matches = re.findall(r"\\boxed\{([^}]+)\}", cot_text)
        return matches[-1].strip() if matches else "nan"

    def generate_cot(self, prompt: str) -> str:
        """Главный маршрутизатор."""
        prompt = str(prompt)
        query_match = re.search(r"determine the result for:\s*([^\n]+)", prompt, re.IGNORECASE)
        if query_match:
            query_str = query_match.group(1).strip()
        else:
            lines = [line.strip() for line in prompt.split('\n') if line.strip()]
            if lines and '=' not in lines[-1]:
                query_str = lines[-1].replace('Question:', '').strip()
            else:
                return "Parse Error: Target not found."
            
        if not query_str:
            return "Parse Error: Empty target."
        
        query_clean = query_str.replace(" ", "")
        
        # Маршрутизация: если есть цифры и формат "число-оператор-число" -> Numeric
        if re.search(r'\d', query_clean) and self._numeric_re.fullmatch(query_clean):
            return self._solve_numeric(prompt, query_clean)
        else:
            return self._solve_symbolic(prompt, query_clean)

    # ==========================================
    # БЛОК ЧИСЛЕННОГО РЕШЕНИЯ (МАТРИЧНЫЙ СКАНЕР)
    # ==========================================

    def _get_operand_configs(self, sa: str, sb: str) -> Dict[str, Tuple[int, int, str, str]]:
        """4 базовые конфигурации операндов."""
        return {
            "fwd": (int(sa), int(sb), sa, sb),
            "rev_digits": (int(self._rev(sa)), int(self._rev(sb)), self._rev(sa), self._rev(sb)),
            "swap_ops": (int(sb), int(sa), sb, sa),
            "swap_rev": (int(self._rev(sb)), int(self._rev(sa)), self._rev(sb), self._rev(sa))
        }

    def _get_operations(self, a: int, b: int, sa: str, sb: str) -> Dict[str, int]:
        """Расширенный пул скрытых математических операций."""
        ops = {
            "add": a + b,
            "sub": a - b,
            "rev_sub": b - a,
            "mul": a * b,
            "cat": int(sa + sb) if sa + sb != "" and len(sa + sb) < 15 else 0,
            "rev_cat": int(sb + sa) if sb + sa != "" and len(sb + sa) < 15 else 0,
            
            "add1": a + b + 1,
            "addm1": a + b - 1,
            "mul1": a * b + 1,
            "mulm1": a * b - 1,
            "sub1": a - b + 1,
            "subm1": a - b - 1,
            
            "muladd_a": a * b + a,
            "mulsub_a": a * b - a,
            "muladd_b": a * b + b,
            "mulsub_b": a * b - b,
            
            "abs_diff": abs(a - b),
            "neg_abs_diff": -abs(a - b),
        }
        
        if b != 0:
            ops["div"] = a // b
            ops["mod"] = a % b
        if a != 0:
            ops["rev_div"] = b // a
            ops["rev_mod"] = b % a
        if a != 0 and b != 0:
            ops["max_mod_min"] = max(a, b) % min(a, b)

        # Поразрядные операции (только для двузначных чисел)
        sa_c, sb_c = sa.lstrip("-"), sb.lstrip("-")
        if len(sa_c) == 2 and len(sb_c) == 2:
            try:
                d1, d2, d3, d4 = int(sa_c[0]), int(sa_c[1]), int(sb_c[0]), int(sb_c[1])
                ops["digit_abs_diff"] = int(str(abs(d1 - d3)) + str(abs(d2 - d4)))
                ops["digit_add_mod10"] = int(str((d1 + d3) % 10) + str((d2 + d4) % 10))
                ops["digit_sub_mod10"] = int(str((d1 - d3) % 10) + str((d2 - d4) % 10))
                ops["cross_mul"] = d1 * d3 + d2 * d4
                ops["cross_mul_rev"] = d1 * d4 + d2 * d3
                ops["digit_mul"] = int(str(d1 * d3) + str(d2 * d4))
                ops["digit_mul_rev"] = int(str(d1 * d4) + str(d2 * d3))
                ops["digit_sum_diff"] = (d1 + d2) - (d3 + d4)
                ops["digit_sum_sum"] = (d1 + d2) + (d3 + d4)
                ops["det"] = d1 * d4 - d2 * d3
                ops["abs_det"] = abs(d1 * d4 - d2 * d3)
            except ValueError:
                pass
        return ops

    def _get_formats(self, val: int) -> Dict[str, str]:
        """Форматирование финального результата (Padding, Sums, Reversals)."""
        sval = str(val)
        abs_val = abs(val)
        s_abs = str(abs_val)
        
        d_sum = sum(int(d) for d in s_abs) if len(s_abs) < 20 else 0
        d_prod = 1
        if len(s_abs) < 20:
            for d in s_abs: 
                d_prod *= int(d)
                
        formats = {
            "raw": sval,
            "rev": "-" + s_abs[::-1] if val < 0 else s_abs[::-1],
            "abs": s_abs,
            "dsum": str(d_sum),
            "dprod": str(d_prod),
            
            "zpad2": f"{val:02d}" if val >= 0 else f"-{abs_val:02d}",
            "zpad3": f"{val:03d}" if val >= 0 else f"-{abs_val:03d}",
            "zpad4": f"{val:04d}" if val >= 0 else f"-{abs_val:04d}",
            
            "rev_zpad2": f"{val:02d}"[::-1].replace("-", "") if val >= 0 else "-" + f"{abs_val:02d}"[::-1],
            "rev_zpad3": f"{val:03d}"[::-1].replace("-", "") if val >= 0 else "-" + f"{abs_val:03d}"[::-1],
            
            "first_digit": sval[0] if val >= 0 else "-" + s_abs[0],
            "last_digit": sval[-1] if val >= 0 else "-" + s_abs[-1],
            
            "x10": str(val * 10),
            "x100": str(val * 100),
        }
        return formats

    def _solve_numeric(self, prompt: str, query_str: str) -> str:
        raw_lines = re.findall(r"([^\n=]+?)\s*=\s*([^\n]+)", prompt)
        parsed = []
        for lhs, rhs in raw_lines:
            m = self._numeric_re.fullmatch(lhs.strip())
            if m:
                try:
                    int(m.group(1))
                    int(m.group(3))
                    parsed.append((m.group(1), m.group(2).strip(), m.group(3), rhs.strip()))
                except ValueError:
                    pass

        if not parsed:
            return "Parse Error: No valid numeric examples."

        by_op = defaultdict(list)
        for a, op, b, out in parsed:
            by_op[op].append((a, b, out))

        qm = self._numeric_re.fullmatch(query_str)
        if not qm: return f"Parse Error: Invalid numeric query format -> {query_str}"
        qa, q_op, qb = qm.group(1), qm.group(2).strip(), qm.group(3)

        lines = [
            "Let's solve this numerical puzzle by scanning the matrix of possible rules.", 
            "\n**Step 1: Analyze formatting & extract values**"
        ]
        
        found_rules = {}
        
        # Получаем списки всех возможных операций и форматов для итерации
        op_names = list(self._get_operations(1, 1, "1", "1").keys())
        fmt_names = list(self._get_formats(1).keys())

        for op_char, group in by_op.items():
            # Определяем формат отрицательных чисел (например, '17/' вместо '-17')
            any_neg_suffix = False
            any_op_suffix = False
            any_op_prefix = False

            for _, _, out in group:
                if op_char != "-":
                    if out.endswith("-") and len(out) > 1: any_neg_suffix = True
                    if out.endswith(op_char) and len(out) > len(op_char): any_op_suffix = True
                    if out.startswith(op_char) and len(out) > len(op_char): any_op_prefix = True

            neg_fmt = "standard"
            if any_op_suffix: neg_fmt = "op_suffix"
            elif any_op_prefix: neg_fmt = "op_prefix"
            elif any_neg_suffix: neg_fmt = "neg_suffix"

            # Нормализация ответов примеров перед сканированием
            transformed = []
            for a, b, out in group:
                t_out = out
                if neg_fmt == "op_suffix" and out.endswith(op_char):
                    t_out = "-" + out[:-len(op_char)]
                elif neg_fmt == "op_prefix" and out.startswith(op_char):
                    t_out = "-" + out[len(op_char):]
                elif neg_fmt == "neg_suffix" and out.endswith("-"):
                    t_out = "-" + out[:-1]
                transformed.append((a, b, t_out))

            # СКАНИРОВАНИЕ
            found = None
            for op_config in ["fwd", "rev_digits", "swap_ops", "swap_rev"]:
                for op_name in op_names:
                    for out_fmt in fmt_names:
                        all_pass = True
                        for ax, bx, exp_norm in transformed:
                            cfg = self._get_operand_configs(ax, bx)[op_config]
                            ops = self._get_operations(*cfg)
                            
                            if op_name not in ops:
                                all_pass = False; break
                                
                            val = ops[op_name]
                            fmts = self._get_formats(val)
                            
                            if fmts[out_fmt] != exp_norm:
                                all_pass = False; break
                                
                        if all_pass:
                            found = FoundRule(op_config, op_name, out_fmt, neg_fmt, op_char)
                            break
                    if found: break
                if found: break
                
            if found:
                found_rules[op_char] = found

        if q_op not in found_rules:
            if found_rules: q_op = list(found_rules.keys())[0] # Fallback на первое известное правило
            else: return "Algorithmic Error: #STOP:SCAN_LIMIT (Rule not found)."

        rule = found_rules[q_op]
        lines.append(f"Match locked! Config: `{rule.op_config}`, Operation: `{rule.op_name}`, Output Format: `{rule.out_fmt}`, Negative Override: `{rule.neg_fmt}`.")

        lines.append("\n**Step 2: Apply combo to target**")
        cfg = self._get_operand_configs(qa, qb)[rule.op_config]
        val = self._get_operations(*cfg)[rule.op_name]
        final_str = self._get_formats(val)[rule.out_fmt]
        
        # Применение форматирования отрицательных чисел (восстановление оператора)
        if final_str.startswith("-"):
            if rule.neg_fmt == "op_suffix": final_str = final_str[1:] + rule.op_char
            elif rule.neg_fmt == "op_prefix": final_str = rule.op_char + final_str[1:]
            elif rule.neg_fmt == "neg_suffix": final_str = final_str[1:] + "-"

        lines.append(f"Result evaluated to {val}, formatted as {final_str}.")
        lines.append(f"The answer is \\boxed{{{final_str}}}")
        return "\n".join(lines)

    # ==========================================
    # БЛОК СИМВОЛЬНОГО РЕШЕНИЯ (CRYPTARITHM + CONCAT)
    # ==========================================

    def _solve_symbolic(self, prompt: str, query_str: str) -> str:
        raw_lines = re.findall(r"([^\n=]+?)\s*=\s*([^\n]+)", prompt)
        parsed_exs = []
        
        for lhs, rhs in raw_lines:
            lhs_c = lhs.replace(" ", "")
            rhs_c = rhs.strip()
            
            if len(lhs_c) >= 3:
                mid = len(lhs_c) // 2
                a, op, b = lhs_c[:mid], lhs_c[mid], lhs_c[mid+1:]
                
                is_neg = False
                if rhs_c.startswith("-") and len(rhs_c) > 1:
                    is_neg = True
                    rhs_c = rhs_c[1:]
                    
                parsed_exs.append({"a": a, "op": op, "b": b, "out": rhs_c, "is_neg": is_neg})
            
        query_clean = query_str.replace(" ", "")
        if not parsed_exs or len(query_clean) < 3:
            return "Parse Error: Invalid symbolic format or query too short."

        mid = len(query_clean) // 2
        q_a, q_op, q_b = query_clean[:mid], query_clean[mid], query_clean[mid+1:]

        unique_syms = set(q_a + q_b)
        for eq in parsed_exs:
            unique_syms.update(list(eq['a'] + eq['b'] + eq['out']))
        
        unique_syms = list(unique_syms)
        lines = ["Let's solve this symbolic logic puzzle step-by-step.", "\n**Step 1: Extract equations and symbols**"]
        
        # Попытка решения через шифр подстановки (бэктрекинг)
        if 0 < len(unique_syms) <= 10:
            lines.append("\n**Step 2: Solve as Cryptarithmetic**")
            compiled = []
            valid_math_ops = {'+', '-', '*', '/'}
            for eq in parsed_exs:
                if eq['op'] not in valid_math_ops: continue
                try:
                    i1 = [unique_syms.index(c) for c in eq['a']]
                    i2 = [unique_syms.index(c) for c in eq['b']]
                    ir = [unique_syms.index(c) for c in eq['out']]
                    compiled.append((i1, eq['op'], i2, ir, eq['is_neg']))
                except ValueError:
                    pass
                
            if compiled:
                found_perm = None
                for perm in itertools.permutations((0,1,2,3,4,5,6,7,8,9), len(unique_syms)):
                    valid = True
                    for i1, op, i2, ir, is_neg in compiled:
                        # Запрет ведущих нулей для многозначных чисел
                        if len(i1) > 1 and perm[i1[0]] == 0: valid = False; break
                        if len(i2) > 1 and perm[i2[0]] == 0: valid = False; break
                        if len(ir) > 1 and perm[ir[0]] == 0: valid = False; break
                        
                        v1 = sum(perm[idx] * (10**(len(i1)-1-j)) for j, idx in enumerate(i1))
                        v2 = sum(perm[idx] * (10**(len(i2)-1-j)) for j, idx in enumerate(i2))
                        vr = sum(perm[idx] * (10**(len(ir)-1-j)) for j, idx in enumerate(ir))
                        
                        if is_neg: vr = -vr
                            
                        if op == '+': valid = (v1 + v2 == vr)
                        elif op == '-': valid = (v1 - v2 == vr)
                        elif op == '*': valid = (v1 * v2 == vr)
                        elif op == '/': valid = (v2 != 0 and v1 // v2 == vr and v1 % v2 == 0)
                        
                        if not valid: break
                    if valid:
                        found_perm = perm
                        break
                        
                if found_perm and q_op in valid_math_ops:
                    inv_map = {v: k for k, v in zip(unique_syms, found_perm)}
                    v1 = sum(found_perm[unique_syms.index(c)] * (10**(len(q_a)-1-j)) for j, c in enumerate(q_a))
                    v2 = sum(found_perm[unique_syms.index(c)] * (10**(len(q_b)-1-j)) for j, c in enumerate(q_b))
                    
                    if q_op == '+': ans = v1 + v2
                    elif q_op == '-': ans = v1 - v2
                    elif q_op == '*': ans = v1 * v2
                    elif q_op == '/': ans = v1 // v2 if v2 != 0 else 0
                    
                    ans_str = str(abs(ans))
                    fs = "-" if ans < 0 else ""
                    fs += "".join(inv_map.get(int(d), "?") for d in ans_str)
                    
                    lines.append(f"The answer is \\boxed{{{fs}}}")
                    return "\n".join(lines)

        # Fallback на простую конкатенацию строк
        lines.append("\n**Step 2: Fallback to String Concatenation**")
        concat_types = {}
        for ex in parsed_exs:
            if ex["out"] == (ex["a"] + ex["b"]): concat_types[ex["op"]] = "fwd"
            elif ex["out"] == (ex["b"] + ex["a"]): concat_types[ex["op"]] = "rev"
        
        c_type = concat_types.get(q_op, "fwd")
        ans = (q_a + q_b) if c_type == "fwd" else (q_b + q_a)
        
        lines.append(f"The answer is \\boxed{{{ans}}}")
        return "\n".join(lines)

In [37]:
from pandarallel import pandarallel

pandarallel.initialize(progress_bar=True)

INFO: Pandarallel will run on 24 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [38]:
eq_df = data[data['label'] == 'equations transformation'].copy()
solver = UnifiedEquationsSolver()

eq_df['generated_cot'] = eq_df['prompt'].parallel_apply(solver.generate_cot)
eq_df['computed_answer'] = eq_df['generated_cot'].parallel_apply(solver.extract_answer)

# Если answer является строкой (символы или числа)
eq_df['is_correct'] = eq_df['computed_answer'] == eq_df['answer'].astype(str).str.strip()

accuracy = eq_df['is_correct'].mean()
print(f"Accuracy (Unified Solver): {accuracy * 100:.2f}%")

Accuracy (Unified Solver): 36.21%


In [31]:
# Фильтруем строки с ошибками (nan) и смотрим, на чем именно падает алгоритм
errors = eq_df[eq_df['computed_answer'] == 'nan']
if not errors.empty:
    print("\n--- ПРИМЕРЫ ОШИБОК ПАРСИНГА ИЛИ ГЕНЕРАЦИИ ---")
    print(errors[['prompt', 'generated_cot']].head(10).to_string())

# Если алгоритм выдает ответ, но он не совпадает с таргетом:
wrong_answers = eq_df[(eq_df['computed_answer'] != 'nan') & (~eq_df['is_correct'])]
if not wrong_answers.empty:
    print("\n--- ПРИМЕРЫ НЕВЕРНЫХ РЕШЕНИЙ ---")
    print(wrong_answers[['prompt', 'computed_answer', 'answer']].head(10).to_string())


--- ПРИМЕРЫ ОШИБОК ПАРСИНГА ИЛИ ГЕНЕРАЦИИ ---
                                                                                                                                                                                                                        prompt                                                                                                                                                                                                          generated_cot
384                    In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:\n["+[@ = "})\n][*!% = \\\@\n]"-"% = @\n[}-[\ = -`@\nNow, determine the result for: }\*%]  Let's solve this symbolic logic puzzle step-by-step.\n\n**Step 1: Extract equations and symbols**\n\n**Step 2: Solve as Cryptarithmetic**\n\n**Step 2: Fallback to String Concatenation**\nThe answer is \boxed{}\%]}
532                    In Alice's Wonderland, a secret set of transformation 

In [32]:
errors_df = eq_df[~eq_df['is_correct']]
for idx, row in errors_df.sample(10).iterrows():
    print(f"=== ID: {row['id']} ===")
    print(f"answer:  '{row['answer']}'")
    print(f"Computed:'{row['computed_answer']}'")
    print(f"Prompt: {row['prompt'][-80:]}\n")

=== ID: 2a5e45a4 ===
answer:  '`@}'
Computed:'<@@$'
Prompt: [}*|! = `!@
}(-|< = :
`:-(} = :}
!}\}< = :(
Now, determine the result for: <@\@$

=== ID: 3d2cb38a ===
answer:  '}{'
Computed:'<'
Prompt: = /)
$\*$} = //#<
<{*!\ = //!}
#{*$) = /{$}
Now, determine the result for: <}+#)

=== ID: 6be00ae9 ===
answer:  '401'
Computed:'2101'
Prompt: 41 = 698
61-26 = -64
52*43 = 058
07-66 = -4
Now, determine the result for: 29+11

=== ID: cca882b8 ===
answer:  '>)<|'
Computed:'!{>)'
Prompt: amples:
>{`!{ = >{!#
{|$|| = '<!
[&${& = <'
Now, determine the result for: !{`>)

=== ID: b451be8a ===
answer:  '64'
Computed:'2@'
Prompt:  53@
81<88 = 4851
32<25 = 6911
13<34 = 3331
Now, determine the result for: 22%42

=== ID: 21dff465 ===
answer:  '@9'
Computed:'155'
Prompt: examples:
30?45 = 75
57?67 = 124
40?37 = 77
Now, determine the result for: 73@82

=== ID: 7d9f6c40 ===
answer:  '461'
Computed:'51'
Prompt: 1-81 = -1
97-58 = -6
06-03 = 03
51-72 = -21
Now, determine the result for: 98+47

=== ID: aa8c76a1

### bit manipulation

In [33]:
import re

class BitManipulationSolver:
    """bit_manipulation - SFT Explanatory Edition"""
    
    def __init__(self):
        self.ops = {
            'I': lambda a, b: a, 'NOT': lambda a, b: 1 - a,
            'C0': lambda a, b: 0, 'C1': lambda a, b: 1,
            'AND': lambda a, b: a & b, 'OR': lambda a, b: a | b, 'XOR': lambda a, b: a ^ b,
            'AND-NOT': lambda a, b: a & (1 - b), 'OR-NOT': lambda a, b: a | (1 - b), 'XOR-NOT': lambda a, b: a ^ (1 - b)
        }
        self.SECTION_ORDER = ['I', 'NOT', 'C0', 'C1', 'AND', 'OR', 'XOR', 'AND-NOT', 'OR-NOT', 'XOR-NOT']

    def _get_valid_rules(self, examples, out_idx):
        valid = []
        for op_name in self.SECTION_ORDER:
            op_func = self.ops[op_name]
            if op_name in ['C0', 'C1']:
                if all(int(ex_out[out_idx]) == op_func(0, 0) for _, ex_out in examples):
                    valid.append((op_name, -1, -1))
            elif op_name in ['I', 'NOT']:
                for in1 in range(8):
                    if all(int(ex_out[out_idx]) == op_func(int(ex_in[in1]), 0) for ex_in, ex_out in examples):
                        valid.append((op_name, in1, -1))
            else:
                for in1 in range(8):
                    for in2 in range(8):
                        is_valid = True
                        for ex_in, ex_out in examples:
                            if op_func(int(ex_in[in1]), int(ex_in[in2])) != int(ex_out[out_idx]):
                                is_valid = False
                                break
                        if is_valid:
                            valid.append((op_name, in1, in2))
        return valid

    def _format_op(self, rule):
        op, in1, in2 = rule
        if op in ['C0', 'C1']: return f"constant {op[-1]}"
        if op == 'I': return f"the value at index {in1}"
        if op == 'NOT': return f"the inverted value of index {in1}"
        return f"the {op} operation between index {in1} and index {in2}"

    def generate_cot(self, prompt: str) -> str:
        examples = []
        for line in prompt.split('\n'):
            if '->' in line:
                parts = line.split('->')
                in_str = re.sub(r'[^01]', '', parts[0])
                out_str = re.sub(r'[^01]', '', parts[1])
                if in_str and out_str:
                    examples.append((in_str.zfill(8), out_str.zfill(8)))

        target_match = re.search(r"output for:\s*([01]+)", prompt, re.IGNORECASE)
        if not target_match or len(examples) == 0:
            return "Parse Error"
        
        target_input = target_match.group(1).zfill(8)
        
        cot = [
            "We need to deduce the 8-bit to 8-bit transformation rule from the provided examples.",
            "Instead of guessing the whole expression at once, let's analyze the transformation bit by bit, looking for continuous sequences (strides) where the operation remains the same but the input indices shift by +1.\n"
        ]

        flat_matches = [self._get_valid_rules(examples, i) for i in range(8)]

        # Left Run
        best_left_run = []
        if flat_matches[0]:
            for cand in flat_matches[0]:
                run = [cand]
                op, in1, in2 = cand
                for i in range(1, 8):
                    exp_in1 = (in1 + i) % 8 if in1 != -1 else -1
                    exp_in2 = (in2 + i) % 8 if in2 != -1 else -1
                    if (op, exp_in1, exp_in2) in flat_matches[i]:
                        run.append((op, exp_in1, exp_in2))
                    else:
                        break
                if len(run) > len(best_left_run):
                    best_left_run = run

        # Right Run
        best_right_run = []
        if flat_matches[7]:
            for cand in flat_matches[7]:
                run = [cand]
                op, in1, in2 = cand
                for step in range(1, 8):
                    i = 7 - step
                    exp_in1 = (in1 - step) % 8 if in1 != -1 else -1
                    exp_in2 = (in2 - step) % 8 if in2 != -1 else -1
                    if (op, exp_in1, exp_in2) in flat_matches[i]:
                        run.insert(0, (op, exp_in1, exp_in2))
                    else:
                        break
                if len(run) > len(best_right_run):
                    best_right_run = run

        len_l = len(best_left_run)
        len_r = len(best_right_run)
        
        cot.append(f"Analyzing from the left (bit 0), the longest consistent operation is `{best_left_run[0][0]}` which successfully covers {len_l} bits.")
        cot.append(f"Analyzing from the right (bit 7) backwards, the longest consistent operation is `{best_right_run[-1][0]}` which covers {len_r} bits.\n")

        # Truncation
        if len_l + len_r > 8:
            cot.append("The left and right sequences overlap. We must truncate the shorter sequence to resolve the conflict.")
            if len_r > len_l:
                len_l = 8 - len_r
                best_left_run = best_left_run[:len_l]
                cot.append("Since the right sequence is longer, we truncate the left sequence.")
            else:
                len_r = 8 - len_l
                best_right_run = best_right_run[-len_r:] if len_r > 0 else []
                cot.append("Since the left sequence is longer (or equal), we truncate the right sequence.")
            cot.append("")

        final_rules = [None] * 8
        for i in range(len_l): final_rules[i] = best_left_run[i]
        right_start_idx = 8 - len_r
        for i in range(len_r): final_rules[right_start_idx + i] = best_right_run[i]

        pending = [i for i in range(8) if final_rules[i] is None]
        
        # Filling Holes
        if pending:
            cot.append(f"Bits {pending} are currently unmatched. We will attempt to deduce their rules.")
            anchor_run = best_right_run if len_r > len_l else best_left_run
            anchor_idx = right_start_idx if len_r > len_l else 0
            base_op = anchor_run[0][0]
            
            can_extrapolate = True
            temp_rules = {}
            for p in pending:
                offset = p - anchor_idx
                exp_in1 = (anchor_run[0][1] + offset) % 8 if anchor_run[0][1] != -1 else -1
                exp_in2 = (anchor_run[0][2] + offset) % 8 if anchor_run[0][2] != -1 else -1
                if (base_op, exp_in1, exp_in2) in flat_matches[p]:
                    temp_rules[p] = (base_op, exp_in1, exp_in2)
                else:
                    can_extrapolate = False
                    break
            
            if can_extrapolate:
                cot.append(f"We can successfully extrapolate the `{base_op}` operation from the dominant sequence to fill all missing bits.")
                for p in pending: final_rules[p] = temp_rules[p]
                pending = []
            else:
                perfect_cat = None
                for cat in self.SECTION_ORDER:
                    if all(any(c[0] == cat for c in flat_matches[p]) for p in pending):
                        perfect_cat = cat
                        break
                if perfect_cat:
                    cot.append(f"Extrapolation failed, but we found that a `{perfect_cat}` operation perfectly fits all remaining bits independently.")
                    for p in pending:
                        final_rules[p] = next(c for c in flat_matches[p] if c[0] == perfect_cat)
                    pending = []

        if pending:
            cot.append("No unified pattern fits the remaining bits. We will apply the best local operation or default to 1 as a fallback.")
            for p in pending:
                if flat_matches[p]:
                    final_rules[p] = flat_matches[p][0]
                else:
                    final_rules[p] = ('C1', -1, -1)
        
        cot.append("\nNow, we apply the final derived rules mapping to the target input string: " + target_input)
        target_output = ""
        for i in range(8):
            op, in1, in2 = final_rules[i]
            val1 = int(target_input[in1]) if in1 != -1 else 0
            val2 = int(target_input[in2]) if in2 != -1 else 0
            res = self.ops[op](val1, val2)
            target_output += str(res)
            
            explanation = self._format_op(final_rules[i])
            cot.append(f"  Bit {i}: Use {explanation} -> {res}")

        cot.append(f"\nThe final answer is {target_output}.")
        return "\n".join(cot)

    def extract_answer(self, cot_text: str) -> str:
        if not cot_text or "Error" in cot_text:
            return None
        match = re.search(r"The final answer is ([01]{8})\.", cot_text)
        return match.group(1) if match else None

In [34]:
bit_df = data[data['label'] == 'bit manipulation'].copy()
solver = BitManipulationSolver()

bit_df['generated_cot'] = bit_df['prompt'].apply(solver.generate_cot)
bit_df['computed_answer'] = bit_df['generated_cot'].apply(solver.extract_answer)
bit_df['is_correct'] = bit_df['computed_answer'].astype(str).str.strip() == bit_df['answer'].astype(str).str.strip()

accuracy = bit_df['is_correct'].mean()
print(f"Accuracy by 'bit manipulation': {accuracy * 100:.2f}%")

Accuracy by 'bit manipulation': 84.33%


In [35]:
errors_df = bit_df[~bit_df['is_correct']]
for idx, row in errors_df.sample(25).iterrows():
    print(f"=== ID: {row['id']} ===")
    print(f"answer:  '{row['answer']}'")
    print(f"Computed:'{row['computed_answer']}'")
    print(f"Prompt: {row['prompt'][-80:]}\n")

=== ID: a8f2c2b9 ===
answer:  '10011111'
Computed:'10111111'
Prompt: 100010 -> 10111000
00101100 -> 00011111

Now, determine the output for: 01111100

=== ID: 6abc8047 ===
answer:  '00101001'
Computed:'10101001'
Prompt: 100000 -> 01110000
10101111 -> 11010111

Now, determine the output for: 01010011

=== ID: 44fb2f96 ===
answer:  '11001110'
Computed:'11111110'
Prompt: 010000 -> 11011101
01010100 -> 11011101

Now, determine the output for: 00011000

=== ID: 4fb6838e ===
answer:  '10000101'
Computed:'01000101'
Prompt: 011011 -> 10000010
01001100 -> 00000010

Now, determine the output for: 10101011

=== ID: b158ab98 ===
answer:  '00000001'
Computed:'10000001'
Prompt: 011000 -> 00000110
10100011 -> 10000000

Now, determine the output for: 10100110

=== ID: 6cfe5536 ===
answer:  '01000010'
Computed:'01000110'
Prompt: 000001 -> 00100010
00011000 -> 00001100

Now, determine the output for: 11010100

=== ID: 05ca617c ===
answer:  '11011011'
Computed:'01011011'
Prompt: 111001 -> 01011100
010100

In [39]:
import pandas as pd

In [40]:
data = pd.read_csv("../data/solved/train-cot.csv")

In [42]:
from transformers import AutoTokenizer

In [44]:
tokenizer = AutoTokenizer.from_pretrained("nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16")

In [48]:
data.generated_cot.apply(tokenizer.encode).apply(len).describe()

count    9500.000000
mean      287.391263
std       156.030213
min       110.000000
25%       169.000000
50%       235.000000
75%       323.000000
max       670.000000
Name: generated_cot, dtype: float64